# Exp 6a: FP32 dose screening

Finds the smallest fine-tuning dose (learning rate x steps) that brings the membership AUC of the deleted nodes back to the level of a retrained model. No quantization in this notebook.

Cora and CiteSeer, 2-layer GCN, 10% of nodes labelled. Selection seeds 20-24, evaluation seeds 30-34. The pre-registration file is written to the output folder before anything is trained.

Runs on CPU in about 15 min. `SMOKE_TEST = True` gives a quick check (numbers not meaningful).

In [ ]:
# 0. Install (Colab and Kaggle already have PyTorch)
!pip install -q torch_geometric

## 1. Configuration

In [ ]:
import os, json, time, random, shutil, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.utils import subgraph
from torch_geometric.nn import GCNConv
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

SMOKE_TEST = False          # True = tiny, fast test run (results are NOT meaningful)
SAVE_TO_DRIVE = False       # Colab only: True saves results in Google Drive (survives disconnects)

if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/exp6_fp32_screening'
else:
    OUT_DIR = 'exp6_fp32_screening' if not SMOKE_TEST else 'exp6_smoke_test'
os.makedirs(f'{OUT_DIR}/per_seed', exist_ok=True); os.makedirs(f'{OUT_DIR}/weights', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG = dict(
    datasets=['Cora', 'CiteSeer'],
    hidden=128, dropout=0.5, lr_train=0.01, weight_decay=5e-4, epochs=200,      # frozen from Experiment 5
    train_frac=0.10, delete_frac=0.10,
    ft_lrs=[0.001, 0.005, 0.01], doses=[1, 2, 5, 10, 20, 50, 100, 200],
    selection_seeds=[20, 21, 22, 23, 24], evaluation_seeds=[30, 31, 32, 33, 34],
    min_seeds_agree=4, utility_tolerance=0.02, small_update_threshold=0.5,
)
if SMOKE_TEST:
    CFG.update(epochs=20, doses=[1, 5, 20], selection_seeds=[20, 21], evaluation_seeds=[30, 31], min_seeds_agree=1)
    print('SMOKE TEST: tiny settings, results are not meaningful.')

def log(msg):
    line = f'[{datetime.datetime.now():%H:%M:%S}] {msg}'
    print(line, flush=True)
    with open(f'{OUT_DIR}/progress.log', 'a') as f: f.write(line + '\n')

json.dump(CFG, open(f'{OUT_DIR}/config.json', 'w'), indent=2)
log(f'Device: {DEVICE} | results folder: {OUT_DIR}')

## 2. Pre-registration (written and saved before any result)

In [ ]:
PREREG = f"""# Experiment 6: minimum effective fine-tuning unlearning update (FP32 screening only). Pre-registration
Saved: {datetime.datetime.now():%Y-%m-%d %H:%M:%S} (before any Experiment 6 result in this run). Quantization is NOT part of this stage.

## Frozen settings (from Experiment 5; no further architecture selection)
Datasets: Cora and CiteSeer (Planetoid, row-normalised features). Model: 2-layer GCN, hidden {CFG['hidden']}, dropout {CFG['dropout']};
Adam lr {CFG['lr_train']}, weight decay {CFG['weight_decay']}, {CFG['epochs']} epochs, last epoch used (no checkpoint selection).
Training nodes: random {int(100*CFG['train_frac'])} % of nodes. Deletion: {int(100*CFG['delete_frac'])} % of the training nodes.
Implementation: sparse-input version of the standard PyG GCN (checked against GCNConv in section 3).

## Seeds
Selection seeds {CFG['selection_seeds']}: used only to choose the dose. Evaluation seeds {CFG['evaluation_seeds']} (fresh): used only to
judge the frozen dose. Each seed draws its own training nodes, deleted nodes, held-out split and initialisations.

## Models per seed
* Original: all training labels, full graph.
* Retrain-A, Retrain-B: deleted nodes' labels removed AND their edges removed from the graph; two different initialisations.
* Fine-tuning unlearning (FT): start from the Original; loss on retained training labels only; message passing on the same
  reduced graph as Retrain; Adam (weight decay {CFG['weight_decay']}), dropout on.
  FT learning rates {CFG['ft_lrs']} x doses (optimizer steps) {CFG['doses']}.
* Drift control: identical fine-tuning on the full training data and full graph (nothing deleted), same grid.

## Measurements (identical inference conditions)
* Membership: every model is queried on the same full graph; score = probability of the true label; members = all deleted
  nodes D; non-members = held-out nodes H_eval (held-out nodes split into H_cal / H_eval per seed).
  AUC, advantage 2|AUC - 0.5|, distance to Retrain-A = |AUC - AUC(Retrain-A)|.
* Paired improvement over Original = dist(Original) - dist(config), same D and H_eval.
* Calibrated attack: threshold chosen on retained training nodes vs H_cal only; TPR(D) - FPR(H_eval).
* Utility: accuracy on held-out nodes using the reduced graph (graph after deletion).
* Update size: ||W - W_Original||; median |dW| / quantization step and share of |dW| > step/2 for INT4 and INT3,
  per-channel and per-tensor, with the step taken from the Original's weights (frozen scales).

## Selection rule (selection seeds only)
band = 2 x RMS over selection seeds of (AUC(Retrain-A) - AUC(Retrain-B)).
Precondition: the Original must clearly fail the band: mean dist(Original) > band AND dist(Original) > band on >= {CFG['min_seeds_agree']} seeds.
A configuration passes if (a) mean dist(config) <= band; (b) paired improvement over Original > 0 on >= {CFG['min_seeds_agree']} seeds;
(c) mean utility >= mean utility(Original) - {CFG['utility_tolerance']}. Among passing configurations choose the smallest mean update norm.
If none passes: "fine-tuning does not reach Retrain"; no dose is frozen for that dataset.

## Evaluation rule (evaluation seeds, frozen dose; all doses are still reported)
Success if: the 95 % t-interval of the paired improvement over Original is above 0; improvement > 0 on >= {CFG['min_seeds_agree']} seeds;
mean dist(frozen dose) <= band (band from the selection seeds); mean utility >= utility(Original) - {CFG['utility_tolerance']};
and the Original fails the band on the evaluation seeds (mean dist(Original) > band).

## Update-size classification
"Small" update: median |dW| / INT4 per-channel step (frozen Original scale) < {CFG['small_update_threshold']} for the frozen dose.

## Decision after this stage
* Success with a small update -> full quantization matrix (separate, new pre-registration).
* Success with only a substantial update -> quantize the smallest successful dose and measure update survival.
* No success on the evaluation seeds -> stop and investigate the unlearning procedure.
"""
path = f'{OUT_DIR}/exp6_preregistration.md'
if os.path.exists(path):
    log('Pre-registration already saved earlier: keeping the original file unchanged.')
else:
    open(path, 'w').write(PREREG); log('Pre-registration saved.')
print(open(path).read())

## 3. Model
A sparse-input implementation of the standard PyG GCN: each layer computes Â·(X Wᵀ) + b with Â = D^-1/2 (A + I) D^-1/2,
exactly like `GCNConv`. Dropout on the sparse input only touches its non-zero entries, which is the same dropout on a
bag-of-words matrix, just faster. The cell checks the outputs against `GCNConv` with identical weights.

In [ ]:
def norm_adj(edge_index, n):
    ei, w = gcn_norm(edge_index, None, n, add_self_loops=True)
    return torch.sparse_coo_tensor(torch.stack([ei[1], ei[0]]), w, (n, n)).coalesce().to(DEVICE)

class Layer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o, bias=False); self.bias = nn.Parameter(torch.zeros(o))
        nn.init.xavier_uniform_(self.lin.weight)
    def forward(self, x, A):
        xw = torch.sparse.mm(x, self.lin.weight.t()) if x.is_sparse else x @ self.lin.weight.t()
        return torch.sparse.mm(A, xw) + self.bias

class GCN(nn.Module):
    def __init__(self, i, h, o, p):
        super().__init__(); self.c1, self.c2, self.p = Layer(i, h), Layer(h, o), p
    def forward(self, xs, A):
        if self.training and self.p > 0:
            xs = torch.sparse_coo_tensor(xs.indices(), F.dropout(xs.values(), self.p, True), xs.shape)
        h = F.relu(self.c1(xs, A)); h = F.dropout(h, self.p, self.training)
        return self.c2(h, A)

# equivalence check against PyG GCNConv
_d = Planetoid('data', 'Cora', transform=T.NormalizeFeatures())[0]
_m = GCN(_d.num_features, 128, 7, 0.5).to(DEVICE).eval()
_r1, _r2 = GCNConv(_d.num_features, 128).to(DEVICE), GCNConv(128, 7).to(DEVICE)
with torch.no_grad():
    _m.c1.bias.add_(0.1)
    _r1.lin.weight.copy_(_m.c1.lin.weight); _r1.bias.copy_(_m.c1.bias); _r2.lin.weight.copy_(_m.c2.lin.weight); _r2.bias.copy_(_m.c2.bias)
    diff = (_m(_d.x.to_sparse().to(DEVICE), norm_adj(_d.edge_index, _d.num_nodes))
            - _r2(F.relu(_r1(_d.x.to(DEVICE), _d.edge_index.to(DEVICE))), _d.edge_index.to(DEVICE))).abs().max().item()
log(f'Equivalence check vs PyG GCNConv: max |difference| = {diff:.2e}')
assert diff < 1e-4, 'implementation does not match GCNConv'

## 4. One seed: Original, two Retrains, fine-tuning grid, drift control

In [ ]:
_DATA = {}
def get_data(name):
    if name not in _DATA:
        d = Planetoid('data', name, transform=T.NormalizeFeatures())[0]
        _DATA[name] = dict(d=d, xs=d.x.to_sparse().to(DEVICE), y=d.y.to(DEVICE), A_full=norm_adj(d.edge_index, d.num_nodes))
    return _DATA[name]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def step_stats(W0, W):
    out = {}
    for bits in [4, 3]:
        qmax = 2 ** (bits - 1) - 1
        for scheme in ['per_channel', 'per_tensor']:
            r = []
            for w0, w in zip(W0, W):
                if w0.dim() != 2: continue
                s0 = (w0.abs().amax(1, keepdim=True) if scheme == 'per_channel' else w0.abs().max()) / qmax
                r.append(((w - w0).abs() / s0).flatten())
            r = torch.cat(r)
            out[f'med_upd_over_step_INT{bits}_{scheme}'] = float(r.median())
            out[f'pct_above_half_step_INT{bits}_{scheme}'] = 100 * float((r > 0.5).float().mean())
    return out

def run_seed(ds, seed, save_weights_for=None):
    # save_weights_for: optional (lr, dose) whose FT weights are saved together with the reference models
    f_out = f'{OUT_DIR}/per_seed/{ds}_seed{seed}.csv'
    if os.path.exists(f_out):
        log(f'{ds} seed {seed}: already done, skipping'); return
    t0 = time.time(); G = get_data(ds); d, xs, Y, A_full = G['d'], G['xs'], G['y'], G['A_full']
    N, C = d.num_nodes, int(d.y.max()) + 1; y = d.y.numpy()
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N); n_tr = int(CFG['train_frac'] * N)
    tr, held = np.sort(perm[:n_tr]), np.sort(perm[n_tr:])
    D = np.sort(rng.choice(tr, int(CFG['delete_frac'] * len(tr)), replace=False)); retained = np.setdiff1d(tr, D)
    hp = rng.permutation(held); H_cal, H_eval = hp[: len(hp) // 2], hp[len(hp) // 2:]
    R_cal = rng.choice(retained, size=min(len(retained), len(H_cal)), replace=False)
    keep = torch.ones(N, dtype=torch.bool); keep[torch.from_numpy(D)] = False
    ei_red, _ = subgraph(keep, d.edge_index, num_nodes=N); A_red = norm_adj(ei_red, N)
    m_full = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_full[torch.from_numpy(tr)] = True
    m_red = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_red[torch.from_numpy(retained)] = True

    def train(A, mask, init_seed):
        set_seed(init_seed); m = GCN(d.num_features, CFG['hidden'], C, CFG['dropout']).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=CFG['lr_train'], weight_decay=CFG['weight_decay'])
        for _ in range(CFG['epochs']):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A)[mask], Y[mask]).backward(); opt.step()
        return [p.detach().clone() for p in m.parameters()]

    W0 = train(A_full, m_full, 1000 + seed); WA = train(A_red, m_red, 2000 + seed); WB = train(A_red, m_red, 3000 + seed)
    probe = GCN(d.num_features, CFG['hidden'], C, CFG['dropout']).to(DEVICE)

    @torch.no_grad()
    def evaluate(W):
        for p, w in zip(probe.parameters(), W): p.data = w.clone()
        probe.eval()
        Pf = F.softmax(probe(xs, A_full), 1).cpu().numpy(); conf = Pf[np.arange(N), y]
        Pr = F.softmax(probe(xs, A_red), 1).cpu().numpy()
        auc = roc_auc_score(np.r_[np.ones(len(D)), np.zeros(len(H_eval))], np.r_[conf[D], conf[H_eval]])
        cand = np.unique(np.r_[conf[R_cal], conf[H_cal]])
        thr = cand[int(np.argmax([((conf[R_cal] >= t).mean() + (conf[H_cal] < t).mean()) / 2 for t in cand]))]
        return dict(auc=auc, adv=2 * abs(auc - 0.5), tpr_minus_fpr=(conf[D] >= thr).mean() - (conf[H_eval] >= thr).mean(),
                    util_reduced_graph=float((Pr.argmax(1)[held] == y[held]).mean()), acc_deleted=float((Pf.argmax(1)[D] == y[D]).mean()))

    rows, keep_w = [], {'Original': W0, 'Retrain-A': WA, 'Retrain-B': WB}
    def add(model, lr, dose, W):
        dw = torch.cat([(w - w0).flatten() for w, w0 in zip(W, W0)])
        r = dict(dataset=ds, seed=seed, n_deleted=len(D), n_heldout_eval=len(H_eval), model=model, lr=lr, dose=dose,
                 update_norm=float(dw.norm()), **evaluate(W))
        if model.startswith('FT'): r.update(step_stats(W0, W))
        rows.append(r)
    add('Original', np.nan, 0, W0); add('Retrain-A', np.nan, np.nan, WA); add('Retrain-B', np.nan, np.nan, WB)
    for kind, A, mask in [('FT', A_red, m_red), ('FT full-data control', A_full, m_full)]:
        for lr in CFG['ft_lrs']:
            set_seed(5000 + seed + int(lr * 1e4))
            m = GCN(d.num_features, CFG['hidden'], C, CFG['dropout']).to(DEVICE)
            for p, w in zip(m.parameters(), W0): p.data = w.clone()
            opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
            for step in range(1, max(CFG['doses']) + 1):
                m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A)[mask], Y[mask]).backward(); opt.step()
                if step in CFG['doses']:
                    W = [p.detach().clone() for p in m.parameters()]; add(kind, lr, step, W)
                    if kind == 'FT' and save_weights_for == (lr, step): keep_w[f'FT lr={lr} dose={step}'] = W
    if save_weights_for is not None:      # reference models + frozen dose, for a later quantization stage
        torch.save({**{k: [w.cpu() for w in v] for k, v in keep_w.items()}, 'D': D, 'H_cal': H_cal, 'H_eval': H_eval,
                    'R_cal': R_cal, 'train': tr, 'held': held}, f'{OUT_DIR}/weights/{ds}_seed{seed}.pt')
    pd.DataFrame(rows).to_csv(f_out, index=False)          # checkpoint: one CSV per finished seed
    log(f'{ds} seed {seed}: done in {time.time()-t0:.0f}s ({len(D)} deleted nodes, {len(H_eval)} held-out eval nodes)')

## 5. Stage A: selection seeds

In [ ]:
for ds in CFG['datasets']:
    for seed in CFG['selection_seeds']:
        run_seed(ds, seed)
log('Stage A finished.')

## 6. Selection rule -> freeze one dose per dataset

In [ ]:
def load(seeds):
    fs = [f'{OUT_DIR}/per_seed/{ds}_seed{s}.csv' for ds in CFG['datasets'] for s in seeds]
    return pd.concat([pd.read_csv(f) for f in fs if os.path.exists(f)])

def summarise(d, ds, band):
    r = d[(d.dataset == ds) & d.model.isin(['Original', 'Retrain-A', 'Retrain-B'])].pivot_table(index='seed', columns='model',
                                                                                              values=['auc', 'util_reduced_graph'])
    dist_orig = (r['auc']['Original'] - r['auc']['Retrain-A']).abs()
    ft = d[(d.dataset == ds) & (d.model == 'FT')].copy()
    ft['retA'] = ft.seed.map(r['auc']['Retrain-A']); ft['dist'] = (ft.auc - ft.retA).abs()
    ft['improvement'] = ft.seed.map(dist_orig) - ft.dist; ft['util_orig'] = ft.seed.map(r['util_reduced_graph']['Original'])
    s = ft.groupby(['lr', 'dose']).agg(auc=('auc', 'mean'), dist=('dist', 'mean'), improvement=('improvement', 'mean'),
                                       seeds_improving=('improvement', lambda x: int((x > 0).sum())),
                                       utility=('util_reduced_graph', 'mean'), utility_original=('util_orig', 'mean'),
                                       tpr_minus_fpr=('tpr_minus_fpr', 'mean'), update_norm=('update_norm', 'mean'),
                                       med_step_INT4_pc=('med_upd_over_step_INT4_per_channel', 'mean'),
                                       pct_half_step_INT4_pc=('pct_above_half_step_INT4_per_channel', 'mean')).reset_index()
    s['passes'] = (s.dist <= band) & (s.seeds_improving >= CFG['min_seeds_agree']) & (s.utility >= s.utility_original - CFG['utility_tolerance'])
    ctl = d[(d.dataset == ds) & (d.model == 'FT full-data control')].groupby(['lr', 'dose']).auc.mean().rename('control_auc').reset_index()
    return r, dist_orig, ft, s.merge(ctl, on=['lr', 'dose'])

FROZEN_PATH = f'{OUT_DIR}/frozen_doses.json'
dsel = load(CFG['selection_seeds'])
frozen = json.load(open(FROZEN_PATH)) if os.path.exists(FROZEN_PATH) else None
if frozen is not None:
    log('Frozen doses already exist: they are NOT recomputed.')
else:
    frozen = {}
    for ds in CFG['datasets']:
        r = dsel[(dsel.dataset == ds) & dsel.model.isin(['Retrain-A', 'Retrain-B'])].pivot_table(index='seed', columns='model', values='auc')
        band = float(2 * np.sqrt(((r['Retrain-A'] - r['Retrain-B']) ** 2).mean()))
        ref, dist_orig, ft, s = summarise(dsel, ds, band)
        pre = dist_orig.mean() > band and (dist_orig > band).sum() >= CFG['min_seeds_agree']
        p = s[s.passes].sort_values('update_norm')
        if not pre:
            frozen[ds] = dict(band=band, status='precondition failed: Original does not clearly fail the band')
        elif len(p) == 0:
            frozen[ds] = dict(band=band, status='no configuration passes: fine-tuning does not reach Retrain')
        else:
            b = p.iloc[0]
            frozen[ds] = dict(band=band, status='frozen', lr=float(b.lr), dose=int(b.dose), mean_update_norm=float(b.update_norm),
                              med_step_INT4_per_channel=float(b.med_step_INT4_pc),
                              update_class='small' if b.med_step_INT4_pc < CFG['small_update_threshold'] else 'substantial')
    json.dump(frozen, open(FROZEN_PATH, 'w'), indent=2); log(f'Frozen doses saved: {frozen}')

for ds in CFG['datasets']:
    band = frozen[ds]['band']; ref, dist_orig, ft, s = summarise(dsel, ds, band)
    print(f"\n=== {ds}, selection seeds | band {band:.4f} | Original AUC {ref['auc']['Original'].mean():.3f} | "
          f"Retrain-A {ref['auc']['Retrain-A'].mean():.3f} | dist(Original) {dist_orig.mean():.3f} ===")
    s.to_csv(f'{OUT_DIR}/selection_summary_{ds}.csv', index=False)
    display(s.round(4))
print(json.dumps(frozen, indent=2))

## 7. Stage B: evaluation seeds (fresh)

In [ ]:
for ds in CFG['datasets']:
    fz = frozen[ds]
    swf = (fz['lr'], fz['dose']) if fz.get('status') == 'frozen' else None
    for seed in CFG['evaluation_seeds']:
        run_seed(ds, seed, save_weights_for=swf)
log('Stage B finished.')

## 8. Evaluation rule (frozen dose); all doses reported

In [ ]:
deval = load(CFG['evaluation_seeds']); outcome = {}
for ds in CFG['datasets']:
    fz = frozen[ds]; band = fz['band']
    ref, dist_orig, ft, s = summarise(deval, ds, band)
    s.to_csv(f'{OUT_DIR}/evaluation_summary_all_doses_{ds}.csv', index=False)
    print(f"\n=== {ds}, evaluation seeds | Original AUC {ref['auc']['Original'].mean():.3f} | Retrain-A {ref['auc']['Retrain-A'].mean():.3f} "
          f"| dist(Original) {dist_orig.mean():.3f} vs band {band:.4f} ===")
    display(s.round(4))
    if fz.get('status') != 'frozen':
        outcome[ds] = dict(status=fz['status']); print('No frozen dose:', fz['status']); continue
    x = ft[(ft.lr == fz['lr']) & (ft.dose == fz['dose'])].set_index('seed')
    n = len(x); m, sd = x.improvement.mean(), x.improvement.std(ddof=1)
    lo, hi = (m - stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n), m + stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)) if n > 1 else (np.nan, np.nan)
    crit = dict(ci_above_0=bool(lo > 0), improves_on_enough_seeds=bool((x.improvement > 0).sum() >= CFG['min_seeds_agree']),
                within_band=bool(x.dist.mean() <= band), utility_ok=bool(x.util_reduced_graph.mean() >= x.util_orig.mean() - CFG['utility_tolerance']),
                original_fails_band=bool(dist_orig.mean() > band))
    outcome[ds] = dict(lr=fz['lr'], dose=fz['dose'], improvement_mean=float(m), ci95=[float(lo), float(hi)],
                       seeds_improving=f'{int((x.improvement > 0).sum())}/{n}', dist_mean=float(x.dist.mean()), band=band,
                       utility=float(x.util_reduced_graph.mean()), utility_original=float(x.util_orig.mean()),
                       med_step_INT4_per_channel=float(x['med_upd_over_step_INT4_per_channel'].mean()),
                       update_class=fz['update_class'], criteria=crit, success=all(crit.values()))
    print(f"Frozen dose lr={fz['lr']}, {fz['dose']} steps -> improvement {m:.4f} (95% CI {lo:.4f} to {hi:.4f}), "
          f"{outcome[ds]['seeds_improving']} seeds improving, dist {x.dist.mean():.4f} vs band {band:.4f}")
    print('Criteria:', crit, '=> SUCCESS' if outcome[ds]['success'] else '=> NOT SUCCESSFUL')
json.dump(outcome, open(f'{OUT_DIR}/evaluation_outcome.json', 'w'), indent=2); log(f'Evaluation outcome saved: {outcome}')

## 9. Plot: membership AUC vs. fine-tuning dose (evaluation seeds)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(CFG['datasets']), figsize=(6 * len(CFG['datasets']), 4))
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CFG['datasets']):
    x = deval[deval.dataset == ds]
    for lr in CFG['ft_lrs']:
        g = x[(x.model == 'FT') & (x.lr == lr)].groupby('dose').auc.agg(['mean', 'std'])
        ax.errorbar(g.index, g['mean'], yerr=g['std'], marker='o', capsize=3, label=f'FT lr={lr}')
    for name, ls in [('Original', '--'), ('Retrain-A', ':')]:
        ax.axhline(x[x.model == name].auc.mean(), ls=ls, c='k', lw=1, label=name)
    if frozen[ds].get('status') == 'frozen':
        ax.axvline(frozen[ds]['dose'], c='grey', lw=0.8)
    ax.set_xscale('log'); ax.set_xlabel('fine-tuning steps (dose)'); ax.set_ylabel('membership AUC (deleted vs held-out)')
    ax.set_title(f'{ds}: evaluation seeds'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_dose_curves.png', dpi=200); plt.show()

## 10. Export everything as a ZIP

In [ ]:
def export():
    base = os.path.basename(OUT_DIR.rstrip('/'))
    shutil.make_archive(base, 'zip', OUT_DIR)
    log(f'ZIP created: {base}.zip ({os.path.getsize(base + ".zip")/1e6:.1f} MB)')
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):            # Kaggle: clickable link instead of a browser download
        from IPython.display import FileLink, display
        print('Kaggle: click the link below, or use the Output panel (/kaggle/working) on the right.')
        display(FileLink(f'{base}.zip'))
        return
    try:
        from google.colab import files; files.download(f'{base}.zip')
    except Exception:
        print('Download the ZIP from the file panel on the left.')
export()

### Emergency export
If the session is about to disconnect before the end, run only the cell below. It zips whatever has been saved so far.

In [ ]:
export()